# publish_vulnerability_package_build
`gold_vuln_package`에서 **이번 배치에서 바뀐 것만** `publish_vulnerability_package`로 MERGE. `gold_vuln_package_build` 완료 후 트리거(§3.9).
CVE 번호, GHSA 번호, 패키지·생태계, 설명, 심각도, 취약 버전범위, 패치 버전, 철회 여부만 내보낸다 — `range_idx`/`source_batch_ids` 같은 lineage 전용 컬럼은 Postgres로 안 나간다.

**주의**: `gold_vuln_package_build`와 동일한 `batch_id`를 파이프라인이 전달해야 "바뀐 것만" 필터링이 맞게 동작한다.

In [ ]:
%run 00_gold_common

In [ ]:
if run_mode == "EVENT":
    raise RuntimeError(
        "publish_vulnerability_package_build는 EVENT 모드에서 실행하지 않습니다 "
        "(KEV 이벤트는 패키지 관계를 바꾸지 않음, 05_Job_Orchestration.md §6.4)."
    )

changed = spark.table(TBL["gold_vuln_package"]).filter(F.col("batch_id") == batch_id)

publish_df = (
    changed
    .withColumnRenamed("normalized_vulnerable_version_range", "vulnerable_version_range")
    .select(
        "vuln_package_key", "ghsa_id", "cve_ids", "package_key", "ecosystem", "package_name",
        "vulnerable_version_range", "first_patched_version",
        "summary", "description", "severity", "cvss_score", "cvss_vector", "cwe_ids",
        "published_at", "withdrawn_at", "is_withdrawn",
    )
    .withColumn("synced_at", F.lit(INGESTED_AT))
)
publish_count = publish_df.count()


In [ ]:
if table_exists(TBL["publish_vulnerability_package"]):
    target = DeltaTable.forName(spark, TBL["publish_vulnerability_package"])
    (
        target.alias("t")
        .merge(publish_df.alias("s"), "t.vuln_package_key = s.vuln_package_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    write_overwrite(publish_df, TBL["publish_vulnerability_package"])

log_batch_run("publish_vulnerability_package_build", "SUCCESS", record_count=publish_count)
print(f"[publish_vulnerability_package_build] rows={publish_count} batch_id={batch_id}")
